# Step 5D.1 — Native Feature Contract Audit (CIC-IDS2017, UNSW-NB15)
Автоматично класифікує КОЖНУ нативну колонку кожного датасету за роллю
(predictor/target/time/identifier/excluded) з причиною — без ручного відкидання
"підозрілих" ознак без пояснення. Людина підтверджує/коригує межові випадки за цим
виводом. Не будує моделі, не фіксує F0 остаточно (лише готує підставу для цього).

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ENRICHED_DIR = Path("/srv/ids_research/data/enriched")

# TIME_COLS — МНОЖИНА (не одна назва) для кожного датасету. Попередня версія перевіряла
# лише одну задану назву (time_col="Stime") і мовчки пропустила парне поле Ltime (кінець
# flow, той самий сирий Unix-timestamp) — реальний баг, знайдений при перегляді Step 5D.1.
DATASETS = {
    "CIC-IDS2017": dict(file="cic_ids2017_with_cve_context.parquet",
                        label_col="attack_label", time_cols={"Timestamp"}),
    "UNSW-NB15": dict(file="unsw_nb15_with_cve_context_dedup.parquet",
                      label_col="attack_cat", time_cols={"Stime", "Ltime"}),
}

loaded = {name: pd.read_parquet(ENRICHED_DIR / cfg["file"]) for name, cfg in DATASETS.items()}
for name, df in loaded.items():
    print(f"{name}: {len(df):,} рядків, {len(df.columns)} колонок")


CIC-IDS2017: 2,827,677 рядків, 96 колонок
UNSW-NB15: 2,059,418 рядків, 62 колонок


In [2]:
# ---------- ШИРОКИЙ пошук потенційно часових полів (не лише вже відомі TIME_COLS) ----------
# Той самий принцип, що й у Step 5C.1: шукаємо ЗА ПАТЕРНОМ назви, а не покладаємось лише
# на список, який самі ж і склали (саме так пропустили Ltime минулого разу). Це діагностика
# для ЛЮДИНИ — не автоматичне рішення про роль.
BROAD_TIME_PATTERN = re.compile(
    r"(time|timestamp|datetime|date|stime|ltime|start|end|seq|order|epoch)", re.IGNORECASE)

for name, cfg in DATASETS.items():
    df = loaded[name]
    candidates = [c for c in df.columns if BROAD_TIME_PATTERN.search(c) and not c.startswith("cve_ctx_")]
    already_known = cfg["time_cols"]
    new_candidates = [c for c in candidates if c not in already_known]
    print(f"\n===== {name}: широкий пошук часових/послідовних полів =====")
    print(f"  Уже в TIME_COLS: {sorted(already_known)}")
    if new_candidates:
        print(f"  ⚠ ЗНАЙДЕНО ЩЕ кандидатів, не включених у TIME_COLS: {new_candidates}")
        for c in new_candidates:
            print(f"    {c}: dtype={df[c].dtype}, приклад={df[c].dropna().iloc[0] if df[c].notna().any() else 'N/A'}")
    else:
        print(f"  Нових кандидатів понад уже відомі TIME_COLS не знайдено.")


===== CIC-IDS2017: широкий пошук часових/послідовних полів =====
  Уже в TIME_COLS: ['Timestamp']
  Нових кандидатів понад уже відомі TIME_COLS не знайдено.

===== UNSW-NB15: широкий пошук часових/послідовних полів =====
  Уже в TIME_COLS: ['Ltime', 'Stime']
  Нових кандидатів понад уже відомі TIME_COLS не знайдено.


In [3]:
# ---------- Класифікація ролі кожної нативної (не cve_ctx_*) колонки ----------
IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)

def classify_column(col, cfg, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)", "виключено з F0 за визначенням; кандидат на F1 risk/context групу"
    if col == cfg["label_col"]:
        return "target", "основна мітка атаки датасету (multiclass) — з неї виводиться binary target"
    if col in cfg["time_cols"]:
        return "time", ("часове поле (множина TIME_COLS, не одна назва) — використовується ДЛЯ "
              "спліту, НЕ як predictor. Сирий Unix-timestamp/datetime у моделі є класичним "
              "джерелом shortcut на календарний період замість змістовної поведінки трафіку.")
    if IP_PATTERN.match(col):
        return "network identifier (IP)", ("висококардинальна мережна ідентифікаційна ознака — виключена "
              "з primary F0 для зменшення dataset-specific identity/memorization effects (не тому, що IP "
              "'не є ознакою' в принципі, а через конкретний ризик пере-навчання на ідентичності хоста).")
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)", ("raw port є категоріальною, НЕ порядковою величиною; "
              "виключено з primary F0, оскільки CTI service mapping (cve_ctx_service) сам похідний саме "
              "з (port, protocol) — залишення сирого порту в F0 частково знецінило б порівняння F0 vs F1, "
              "бо F1 показувала б приріст, який частково вже був би доступний через сирий порт у F0.")
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)", "унікальний ідентифікатор запису, не несе flow-статистики"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata", "додано НАШИМ власним пайплайном (напр. __source_file для UNSW), не нативна flow-ознака"
    if col.lower() == "label":
        return "target-derived (redundant)", "бінарна мітка, вже еквівалентна attack_cat!=Normal — дублює target, не predictor"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)", "протокол — валідна ознака, потребує категоріального кодування пізніше"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)", ("нативне поле виявленого сервісу — ЗМІСТОВНО перетинається з "
              "cve_ctx_service (обидва похідні від порту/протоколу); лишається у F0, але варто це "
              "враховувати при інтерпретації приросту F1")
    return "predictor", "за замовчуванням: числова/категоріальна flow-статистика, немає підстав виключати"

audit_rows = {}
for name, cfg in DATASETS.items():
    df = loaded[name]
    rows = []
    for col in df.columns:
        role, reason = classify_column(col, cfg, df)
        rows.append(dict(dataset=name, column=col, dtype=str(df[col].dtype), role=role, reason=reason))
    audit_rows[name] = pd.DataFrame(rows)
    print(f"\n===== {name}: повний feature contract audit ({len(rows)} колонок) =====")
    display(audit_rows[name])


===== CIC-IDS2017: повний feature contract audit (96 колонок) =====


,dataset,column,dtype,role,reason
0,CIC-IDS2017,flow_id,Int64,identifier (flow ID),"унікальний ідентифікатор запису, не несе flow-..."
1,CIC-IDS2017,source_ip,string,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
2,CIC-IDS2017,source_port,Int64,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
3,CIC-IDS2017,destination_ip,string,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
4,CIC-IDS2017,destination_port,Int64,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
...,...,...,...,...,...
91,CIC-IDS2017,cve_ctx_use_epss,bool,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
92,CIC-IDS2017,cve_ctx_service_max_epss,float64,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
93,CIC-IDS2017,cve_ctx_service_in_kev,bool,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
94,CIC-IDS2017,cve_ctx_n_generic_os_matches,int64,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...



===== UNSW-NB15: повний feature contract audit (62 колонок) =====


,dataset,column,dtype,role,reason
0,UNSW-NB15,srcip,str,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
1,UNSW-NB15,sport,str,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
2,UNSW-NB15,dstip,str,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
3,UNSW-NB15,dsport,str,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
4,UNSW-NB15,proto,str,predictor (categorical),"протокол — валідна ознака, потребує категоріал..."
...,...,...,...,...,...
57,UNSW-NB15,cve_ctx_use_epss,bool,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
58,UNSW-NB15,cve_ctx_service_max_epss,float64,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
59,UNSW-NB15,cve_ctx_service_in_kev,bool,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
60,UNSW-NB15,cve_ctx_n_generic_os_matches,int64,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...


In [4]:
# ---------- Зведення по ролях + попередній F0-кандидат (без остаточної фіксації) ----------
F0_candidates = {}
for name, df_audit in audit_rows.items():
    print(f"\n===== {name}: підсумок по ролях =====")
    print(df_audit["role"].value_counts())

    f0_cols = df_audit.loc[df_audit["role"] == "predictor", "column"].tolist()
    f0_cols += df_audit.loc[df_audit["role"] == "predictor (categorical)", "column"].tolist()
    f0_cols += df_audit.loc[df_audit["role"] == "predictor (overlap risk)", "column"].tolist()
    F0_candidates[name] = f0_cols
    print(f"\nF0-кандидат ({len(f0_cols)} колонок): {f0_cols}")

    excluded = df_audit.loc[~df_audit["role"].isin(
        ["predictor", "predictor (categorical)", "predictor (overlap risk)"]), ["column", "role", "reason"]]
    print(f"\nВиключено з F0 ({len(excluded)} колонок), з причиною:")
    display(excluded)


===== CIC-IDS2017: підсумок по ролях =====
role
predictor                             76
CTI (cve_ctx_*)                       12
network identifier (IP)                2
network categorical feature (port)     2
identifier (flow ID)                   1
predictor (categorical)                1
time                                   1
target                                 1
Name: count, dtype: int64

F0-кандидат (77 колонок): ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', '

,column,role,reason
0,flow_id,identifier (flow ID),"унікальний ідентифікатор запису, не несе flow-..."
1,source_ip,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
2,source_port,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
3,destination_ip,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
4,destination_port,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
6,Timestamp,time,"часове поле (множина TIME_COLS, не одна назва)..."
83,attack_label,target,основна мітка атаки датасету (multiclass) — з ...
84,cve_ctx_service,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
85,cve_ctx_pool_size,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...
86,cve_ctx_context_available,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...



===== UNSW-NB15: підсумок по ролях =====
role
predictor                             39
CTI (cve_ctx_*)                       12
network identifier (IP)                2
network categorical feature (port)     2
time                                   2
predictor (categorical)                1
predictor (overlap risk)               1
target                                 1
target-derived (redundant)             1
provenance metadata                    1
Name: count, dtype: int64

F0-кандидат (41 колонок): ['state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service']

Виключено

,column,role,reason
0,srcip,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
1,sport,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
2,dstip,network identifier (IP),висококардинальна мережна ідентифікаційна озна...
3,dsport,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
28,Stime,time,"часове поле (множина TIME_COLS, не одна назва)..."
29,Ltime,time,"часове поле (множина TIME_COLS, не одна назва)..."
47,attack_cat,target,основна мітка атаки датасету (multiclass) — з ...
48,Label,target-derived (redundant),"бінарна мітка, вже еквівалентна attack_cat!=No..."
49,__source_file,provenance metadata,додано НАШИМ власним пайплайном (напр. __sourc...
50,cve_ctx_service,CTI (cve_ctx_*),виключено з F0 за визначенням; кандидат на F1 ...


In [5]:
# ---------- Перевірка: чи "категоріальні" F0-предиктори НАСПРАВДІ числові (артефакт типів) ----------
# Деякі колонки могли стати string/object через власне виправлення Step 3 (де ми приводили
# ЗМІШАНІ типи до рядкового перед збереженням parquet — проблема PyArrow, не природа даних).
# Перевіряємо КОЖЕН такий предиктор через pd.to_numeric(errors="coerce") і звітуємо повну
# діагностику — рішення "categorical -> numerical" приймається лише за ЯВНИМ підтвердженням
# (без додаткових NaN, значення невід'ємні й виглядають як реальний count/numeric), а не
# автоматично для будь-якої колонки, що технічно конвертується.
numeric_coercion_results = {}
for name, df_audit in audit_rows.items():
    df = loaded[name]
    string_predictors = df_audit[
        (df_audit["role"].isin(["predictor", "predictor (categorical)", "predictor (overlap risk)"])) &
        (~df_audit["dtype"].str.contains("int|float", case=False, na=False))
    ]["column"].tolist()

    print(f"\n===== {name}: перевірка string/object F0-предикторів через pd.to_numeric =====")
    if not string_predictors:
        print("  Немає предикторів з нечисловим типом для перевірки.")
        continue

    for col in string_predictors:
        original = df[col]
        coerced = pd.to_numeric(original, errors="coerce")
        n_new_nan = int((coerced.isna() & original.notna()).sum())
        n_total = len(original)
        unique_vals = original.dropna().unique()
        n_unique = len(unique_vals)

        looks_numeric_count = (n_new_nan == 0) and (n_unique > 2)  # >2, щоб не плутати з бінарними прапорцями
        all_non_negative = bool((coerced.dropna() >= 0).all()) if looks_numeric_count else None

        print(f"\n  {col}:")
        print(f"    втрачено при конвертації (нові NaN): {n_new_nan} з {n_total} ({n_new_nan/n_total:.2%})")
        print(f"    унікальних значень: {n_unique}" + (f", приклади: {unique_vals[:5].tolist()}" if n_unique <= 10 else ""))
        if looks_numeric_count:
            print(f"    усі значення невід'ємні: {all_non_negative}")
            verdict = "numerical" if (n_new_nan == 0 and all_non_negative) else "categorical (залишено — не всі умови виконано)"
        else:
            verdict = "categorical (замало унікальних значень або втрата при конвертації — залишено як є)"
        print(f"    ВЕРДИКТ: {verdict}")
        numeric_coercion_results[(name, col)] = verdict


===== CIC-IDS2017: перевірка string/object F0-предикторів через pd.to_numeric =====

  protocol:
    втрачено при конвертації (нові NaN): 2827677 з 2827677 (100.00%)
    унікальних значень: 3, приклади: ['tcp', 'other', 'udp']
    ВЕРДИКТ: categorical (замало унікальних значень або втрата при конвертації — залишено як є)

===== UNSW-NB15: перевірка string/object F0-предикторів через pd.to_numeric =====

  proto:
    втрачено при конвертації (нові NaN): 2059418 з 2059418 (100.00%)
    унікальних значень: 135
    ВЕРДИКТ: categorical (замало унікальних значень або втрата при конвертації — залишено як є)

  state:
    втрачено при конвертації (нові NaN): 2059418 з 2059418 (100.00%)
    унікальних значень: 16
    ВЕРДИКТ: categorical (замало унікальних значень або втрата при конвертації — залишено як є)

  service:
    втрачено при конвертації (нові NaN): 2059418 з 2059418 (100.00%)
    унікальних значень: 13
    ВЕРДИКТ: categorical (замало унікальних значень або втрата при конвертації —

In [6]:
# ---------- UNSW ct_ftp_cmd raw-value audit (детальний розбір, не лише nunique) ----------
import pandas as pd
import numpy as np

col = "ct_ftp_cmd"
raw = loaded["UNSW-NB15"][col]

print(f"===== {col}: повний raw-value audit =====")
print(f"Загальна кількість рядків: {len(raw):,}")
print(f"dtype: {raw.dtype}")
print(f"NaN/None (pandas): {raw.isna().sum():,} ({raw.isna().mean():.2%})")

print(f"\nПовний value_counts (усі унікальні значення, не лише перші 5):")
vc = raw.value_counts(dropna=False)
print(vc)

# Явно перевіряємо підозрілі "missing-like" представлення: порожній рядок, пробіл, "-", "?"
missing_like_patterns = ["", " ", "-", "?", "NA", "N/A", "null", "None"]
print(f"\nЧастка кожного missing-like патерну (якщо є серед значень):")
for pat in missing_like_patterns:
    cnt = (raw == pat).sum()
    if cnt:
        print(f"  {pat!r}: {cnt:,} ({cnt/len(raw):.2%})")

# pd.to_numeric з детальним розбором: що САМЕ не конвертується
coerced = pd.to_numeric(raw, errors="coerce")
newly_nan_mask = coerced.isna() & raw.notna()
non_numeric_values = raw[newly_nan_mask]
print(f"\nЗначення, що НЕ конвертуються в число (raw.notna(), але coerced.isna()): "
      f"{len(non_numeric_values):,} ({len(non_numeric_values)/len(raw):.2%})")
print(f"Унікальні НЕконвертовані значення з кількістю:")
print(non_numeric_values.value_counts(dropna=False))

# Для значень, що ТАКИ конвертуються — статистика
numeric_ok = coerced.dropna()
if len(numeric_ok):
    print(f"\nСтатистика серед УСПІШНО конвертованих значень ({len(numeric_ok):,} з {len(raw):,}):")
    print(numeric_ok.describe())
    print(f"Усі невід'ємні: {(numeric_ok >= 0).all()}")

# Підсумковий вердикт із чітким обґрунтуванням (не лише nunique)
print("\n" + "=" * 70)
print("ВЕРДИКТ")
print("=" * 70)
if len(non_numeric_values) == 0:
    print("Усі значення конвертуються — ct_ftp_cmd є ЧИСТИМ numerical count.")
else:
    unique_non_numeric = non_numeric_values.unique()
    all_look_like_missing = all(str(v).strip() in missing_like_patterns for v in unique_non_numeric)
    if all_look_like_missing:
        print(f"Нечислові значення ({list(unique_non_numeric)}) виглядають як missing-плейсхолдер, "
              f"НЕ окремі номінальні категорії.")
        print("Рекомендація для Step 5D.2: ct_ftp_cmd -> numerical count, "
              "missing-плейсхолдер -> NaN (чи окремий is_ftp_flow-прапорець), "
              "НЕ трактувати як categorical.")
    else:
        print(f"Нечислові значення ({list(unique_non_numeric)}) НЕ виглядають однорідно як "
              f"missing-плейсхолдер — потребують ручного розгляду перед рішенням.")
        print("Залишаємо categorical до подальшого уточнення.")

===== ct_ftp_cmd: повний raw-value audit =====
Загальна кількість рядків: 2,059,418
dtype: str
NaN/None (pandas): 0 (0.00%)

Повний value_counts (усі унікальні значення, не лише перші 5):
ct_ftp_cmd
     1014168
0    1005479
1      38853
2        263
4        250
3        206
5        105
6         91
8          3
Name: count, dtype: int64

Частка кожного missing-like патерну (якщо є серед значень):
  ' ': 1,014,168 (49.25%)

Значення, що НЕ конвертуються в число (raw.notna(), але coerced.isna()): 1,014,168 (49.25%)
Унікальні НЕконвертовані значення з кількістю:
ct_ftp_cmd
    1014168
Name: count, dtype: int64

Статистика серед УСПІШНО конвертованих значень (1,045,250 з 2,059,418):
count    1.045250e+06
mean     4.026979e-02
std      2.190563e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      8.000000e+00
Name: ct_ftp_cmd, dtype: float64
Усі невід'ємні: True

ВЕРДИКТ
Нечислові значення ([' ']) виглядають як missing-плейсхолдер, НЕ окрем

In [7]:
# ---------- Явне попередження щодо межових випадків, які потребують підтвердження людиною ----------
print("=" * 70)
print("МЕТОДОЛОГІЧНО ЧУТЛИВІ ВИПАДКИ (рішення вже прийнято автоматично, потребують підтвердження)")
print("=" * 70)
for name, df_audit in audit_rows.items():
    borderline = df_audit[df_audit["role"].isin(
        ["network categorical feature (port)", "predictor (overlap risk)", "target-derived (redundant)"])]
    if len(borderline):
        print(f"\n{name}:")
        display(borderline[["column", "role", "reason"]])

print("\nℹ Ці рядки є методологічно чутливими випадками: їхнє рішення (port -> excluded,")
print("  service -> included, Label -> excluded) та причина явно зафіксовані в audit-таблиці")
print("  вище. Мають бути ПІДТВЕРДЖЕНІ (або скориговані) перед freeze F0 у Step 5D.2 -")
print("  після підтвердження їх можна фіксувати вже без ambiguity.")

МЕТОДОЛОГІЧНО ЧУТЛИВІ ВИПАДКИ (рішення вже прийнято автоматично, потребують підтвердження)

CIC-IDS2017:


,column,role,reason
2,source_port,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
4,destination_port,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."



UNSW-NB15:


,column,role,reason
1,sport,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
3,dsport,network categorical feature (port),"raw port є категоріальною, НЕ порядковою велич..."
13,service,predictor (overlap risk),нативне поле виявленого сервісу — ЗМІСТОВНО пе...
48,Label,target-derived (redundant),"бінарна мітка, вже еквівалентна attack_cat!=No..."



ℹ Ці рядки є методологічно чутливими випадками: їхнє рішення (port -> excluded,
  service -> included, Label -> excluded) та причина явно зафіксовані в audit-таблиці
  вище. Мають бути ПІДТВЕРДЖЕНІ (або скориговані) перед freeze F0 у Step 5D.2 -
  після підтвердження їх можна фіксувати вже без ambiguity.


In [8]:
# ---------- Фінальний count/type audit F0-кандидата (для Step 5D.2, щоб не помилитись у preprocessing) ----------
for name, df in loaded.items():
    df_audit = audit_rows[name]
    n_native_total = len(df_audit) - (df_audit["role"] == "CTI (cve_ctx_*)").sum()
    n_excluded = ((df_audit["role"] != "predictor") &
                 (df_audit["role"] != "predictor (categorical)") &
                 (df_audit["role"] != "predictor (overlap risk)") &
                 (df_audit["role"] != "CTI (cve_ctx_*)")).sum()
    f0_cols = F0_candidates[name]

    print(f"\n===== {name}: фінальний count/type audit F0 =====")
    print(f"  Нативних колонок (без cve_ctx_*): {n_native_total}")
    print(f"  Виключено (identifier/target/time/provenance/redundant): {n_excluded}")
    print(f"  F0 predictors: {len(f0_cols)}")

    # is_numeric_dtype НЕ ловить boolean як "числовий" для наших цілей (bool трактуємо як
    # категоріальний прапорець), і коректно розпізнає string/category dtype незалежно від
    # того, як саме pandas представляє текстові колонки (object, string[pyarrow] тощо) —
    # на відміну від наївного "dtype == object", яке пропускало сучасні string-dtype.
    categorical_cols, numerical_cols = [], []
    for col in f0_cols:
        coercion_verdict = numeric_coercion_results.get((name, col))
        if coercion_verdict == "numerical":
            # pd.to_numeric підтвердив: жодних нових NaN, значення невід'ємні — реальний
            # числовий count, що потрапив у string-тип через артефакт Step 3, не природа даних.
            numerical_cols.append(col)
        elif df[col].dtype == bool or not pd.api.types.is_numeric_dtype(df[col]):
            categorical_cols.append(col)
        else:
            numerical_cols.append(col)

    print(f"\n  F0 categorical ({len(categorical_cols)}): {categorical_cols}")
    print(f"  F0 numerical ({len(numerical_cols)}): {numerical_cols}")


===== CIC-IDS2017: фінальний count/type audit F0 =====
  Нативних колонок (без cve_ctx_*): 84
  Виключено (identifier/target/time/provenance/redundant): 7
  F0 predictors: 77

  F0 categorical (1): ['protocol']
  F0 numerical (76): ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length